# L5 — Image Segmentation: Follow-Along Examples
**MCS 3950 Computer Vision**

This notebook accompanies **Lecture 5**. It runs on the three PEI aerial clips
you met in AL2.

| Section | Topic |
|---------|-------|
| 0 | Get the imagery |
| 1 | Global thresholding |
| 2 | Otsu — and when its assumption breaks |
| 3 | Adaptive thresholding |
| 4 | Region growing |
| 5 | Morphology, the distance transform, and watershed |
| 6 | Evaluating a segmentation |
| ✏️ | **Try at home** |

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

def show(images, titles, cmap='gray', vmin=0, vmax=255, figw=15):
    """Show a row of images."""
    fig, axes = plt.subplots(1, len(images), figsize=(figw, 4))
    if len(images) == 1:
        axes = [axes]
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(t, fontsize=11)
        ax.axis('off')
    plt.tight_layout(); plt.show()

print("Setup complete.")

---
## 0 · Get the imagery

Same download cell as AL2 — the three clips come from the public course repository.

In [ ]:
# ── Get the aerial imagery ──────────────────────────────────────────
# Downloads three scanned PEI aerial photographs (~3 MB). Safe to re-run.
import io, os, urllib.request, zipfile

DATA_URL = ("https://raw.githubusercontent.com/andrewgodbout/"
            "MCS-3950-F26/main/Notebooks/data/AL2.zip")
NEEDED = ["pei_1935.png", "pei_1958.png", "pei_1968.png"]

if all(os.path.exists(f) for f in NEEDED):
    print("Aerial imagery already present — skipping download.")
else:
    with urllib.request.urlopen(DATA_URL, timeout=60) as resp:
        blob = resp.read()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(".")
    print(f"  {len(blob)/1e6:.1f} MB downloaded and extracted")

# resample all three onto a common 800 x 800 grid at 1 m/px, as in AL2
RES = {1935: 0.5, 1958: 1.1593543264609192, 1968: 0.5}
YEARS = [1935, 1958, 1968]

def to_grid(img, src_res, target=1.0):
    s = src_res / target
    wh = (int(round(img.shape[1]*s)), int(round(img.shape[0]*s)))
    interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, wh, interpolation=interp)

aerial = {y: to_grid(cv2.imread(f"pei_{y}.png", cv2.IMREAD_GRAYSCALE), RES[y])
          for y in YEARS}
side = min(min(a.shape) for a in aerial.values())
aerial = {y: a[:side, :side] for y, a in aerial.items()}

for y in YEARS:
    a = aerial[y]
    print(f"{y}: {a.shape[1]}x{a.shape[0]} px   mean {a.mean():5.1f}   std {a.std():4.1f}")

show([aerial[y] for y in YEARS], [str(y) for y in YEARS])

---
## 1 · Global thresholding

One number for the whole image. The only question is what number.

In [ ]:
img = aerial[1935]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("1935 original"); axes[0].axis('off')

for ax, t in zip(axes[1:], [100, 140, 180]):
    mask = (img > t).astype(np.uint8) * 255
    ax.imshow(mask, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"t = {t}   ({100*(mask>0).mean():.0f}% above)")
    ax.axis('off')
plt.tight_layout(); plt.show()

print("Same image, three thresholds, three completely different answers.")
print("Nothing in the image tells you which one is correct.")

---
## 2 · Otsu — and when its assumption breaks

Otsu chooses `t` by maximising the between-class variance:

$$\sigma^2_{between}(t) = w_0(t)\,w_1(t)\,[\mu_0(t) - \mu_1(t)]^2$$

It assumes the histogram has **two modes with a valley between them**. Let's check
whether that is actually true for our three epochs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, y in zip(axes, YEARS):
    im = aerial[y]
    t, _ = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    h = np.bincount(im.ravel(), minlength=256).astype(float)

    # count modes on a smoothed copy
    hs = np.convolve(h, np.ones(9)/9, mode='same')
    peaks = [i for i in range(5, 251)
             if hs[i] == max(hs[i-5:i+6]) and hs[i] > hs.max()*0.15]

    ax.fill_between(np.arange(256), h, alpha=0.35)
    ax.plot(np.arange(256), h, lw=1.4)
    ax.axvline(t, color='crimson', lw=2, ls='--')
    ax.text(t+5, h.max()*0.88, f"Otsu {t:.0f}", color='crimson', fontweight='bold')
    ax.set_title(f"{y} — {'bimodal' if len(peaks) > 1 else 'UNIMODAL'}  "
                 f"(peaks at {peaks})")
    ax.set_yticks([]); ax.set_xlabel("grey level")
plt.tight_layout(); plt.show()

In [ ]:
# What does that mean in practice?
for y in YEARS:
    im = aerial[y]
    t, mask = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print(f"{y}: Otsu t = {t:5.0f}   ->  {100*(mask>0).mean():5.1f}% labelled foreground")

print()
print("Otsu returned a number for all three. For 1968 that number is meaningless —")
print("there is no valley for it to find, but the function cannot tell you that.")

**Checkpoint.** Otsu never reports failure. Before you trust it on any image,
plot the histogram and look for the valley. If there isn't one, the threshold is
an artifact of the formula, not a property of the scene.

---
## 3 · Adaptive thresholding

Compute a local threshold from a window around each pixel. This is what survives
an illumination gradient across the frame.

In [ ]:
img = aerial[1968]           # the unimodal one — hardest case
t, glob = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adap = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                             cv2.THRESH_BINARY, 51, 5)

show([img, glob, adap],
     ["1968 original", f"global Otsu (t={t:.0f})", "adaptive, 51 px window"])

def quadrant_spread(mask):
    """How much does the foreground fraction vary across the frame?"""
    h, w = mask.shape
    q = [mask[:h//2, :w//2].mean(), mask[:h//2, w//2:].mean(),
         mask[h//2:, :w//2].mean(), mask[h//2:, w//2:].mean()]
    q = [100*v/255 for v in q]
    return q, max(q) - min(q)

for name, m in (("global  ", glob), ("adaptive", adap)):
    q, spread = quadrant_spread(m)
    print(f"{name}: quadrants {[f'{v:.0f}%' for v in q]}   spread {spread:.0f}%")

print()
print("This is the AL2 Checkpoint 3c problem again: a position-independent")
print("transform cannot correct a position-dependent one.")

---
## 4 · Region growing

Start from a seed and expand while neighbours stay within a tolerance.
Watch what the tolerance does.

In [ ]:
def region_grow(img, seed, tol):
    """4-connected region growing from a single seed."""
    h, w = img.shape
    out = np.zeros((h, w), bool)
    out[seed] = True
    stack = [seed]
    value = float(img[seed])
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1,0), (-1,0), (0,1), (0,-1)):
            rr, cc = r+dr, c+dc
            if (0 <= rr < h and 0 <= cc < w and not out[rr, cc]
                    and abs(float(img[rr, cc]) - value) <= tol):
                out[rr, cc] = True
                stack.append((rr, cc))
    return out

img  = aerial[1968]
seed = (400, 400)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, tol in zip(axes, [5, 10, 20, 30]):
    m = region_grow(img, seed, tol)
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB); rgb[m] = [220, 60, 40]
    ax.imshow(rgb); ax.plot(seed[1], seed[0], 'o', color='yellow', ms=6)
    ax.set_title(f"tol = {tol}\n{100*m.mean():.1f}% claimed"); ax.axis('off')
plt.tight_layout(); plt.show()

print("6x the tolerance gives 50x the area. The growth is not gradual —")
print("once the tolerance exceeds the boundary contrast, the region escapes.")

---
## 5 · Morphology, Distance Transform and Watershed

Treat intensity as terrain and flood it. Unguided, every local minimum becomes a
basin — which is far too many.

In [ ]:
img = aerial[1935]
sm  = cv2.GaussianBlur(img, (0, 0), 2)
gx  = cv2.Sobel(sm, cv2.CV_32F, 1, 0, 3)
gy  = cv2.Sobel(sm, cv2.CV_32F, 0, 1, 3)
grad = cv2.normalize(np.hypot(gx, gy), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# count local minima — each would seed its own basin
mins = (grad == cv2.erode(grad, np.ones((3,3), np.uint8))).astype(np.uint8)
n_minima, _ = cv2.connectedComponents(mins)
print(f"local minima in the gradient image: {n_minima:,}")
print(f"-> an unguided watershed returns roughly that many regions")

### 5.1 · Morphology and the distance transform

These are the three tools, written out so you can work
through them at your own pace. None is a segmentation method in its own right —
they are the plumbing that turns "flood the terrain" into "flood it from
sensible starting points". **AL4 on Oct 2 goes further**, with structuring
element shapes, noise cleanup measured against ground truth, and counting
objects that touch.

---

**(a) Erosion, dilation and opening — reshaping a binary mask**

All three slide a small shape (a *structuring element*, here a 5×5 square) over
a black-and-white mask and ask a yes/no question at every pixel.

| | Question asked | Effect on the mask |
|---|---|---|
| **Erosion** | does the 5×5 square fit *entirely* inside the white region? | white regions shrink; anything thinner than 5 px disappears |
| **Dilation** | does the square touch any white at all? | white regions grow by ~2 px on every side |
| **Opening** | erode, then dilate | small specks vanish for good; big regions come back close to their original size |

Opening is the useful one: erosion alone deletes the specks *and* shrinks
everything else, and the dilation afterwards undoes the shrinking for anything
that survived. What was deleted stays deleted — dilation cannot resurrect it.

**(b) The distance transform — how deep inside a region am I?**

`cv2.distanceTransform` replaces every **white** pixel with its distance to the
nearest **black** pixel. Pixels just inside a boundary get small values; pixels
deep in the middle of a large region get large ones. For a disk of radius *r*
the value peaks at exactly *r*, dead centre.

That is the whole trick: **the deepest points are the most confidently
"inside"**, so they make good seeds.

**(c) OpenCV's marker convention**

`cv2.watershed` wants an integer image where

- `0` means *"I don't know — you decide"*,
- any positive integer is a seed label, and every pixel with that label is
  declared to belong to that region,

and it writes `-1` into the pixels where two floods meet. That is why the code
below does `markers + 1` (to lift the background off zero) and then sets the
contested band back to `0`.

In [ ]:
# ── A 60-second demo of the distance transform, on a shape we control ──────
demo = np.zeros((200, 340), np.uint8)
cv2.circle(demo, (113, 100), 60, 255, -1)      # two disks that overlap by 5 px:
cv2.circle(demo, (227, 100), 60, 255, -1)      # ONE connected component

n_cc, _ = cv2.connectedComponents(demo)
d = cv2.distanceTransform(demo, cv2.DIST_L2, 5)
seeds = (d > 0.5*d.max()).astype(np.uint8)
n_seed, _ = cv2.connectedComponents(seeds)

fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
ax[0].imshow(demo, cmap='gray')
ax[0].set_title(f"two touching disks\nconnectedComponents says {n_cc-1}")
im = ax[1].imshow(d, cmap='viridis'); ax[1].set_title("distance transform")
plt.colorbar(im, ax=ax[1], fraction=0.035)
ax[2].plot(d[100, :], lw=2, color='#1A3A5C')
ax[2].axhline(0.5*d.max(), color='#C8962C', ls='--', lw=2)
ax[2].set_title("a slice across the middle"); ax[2].set_xlabel("column")
ax[2].set_ylabel("distance to background")
ax[2].text(6, 0.5*d.max()+2, "cut at 0.5 x max", color='#C8962C', fontsize=10)
ax[3].imshow(seeds, cmap='gray'); ax[3].set_title(f"after the cut: {n_seed-1} seeds")
for a in (ax[0], ax[1], ax[3]): a.axis('off')
plt.tight_layout(); plt.show()

assert n_cc - 1 == 1 and n_seed - 1 == 2, "demo geometry broken"
print(f"The two disks touch, so they are {n_cc-1} connected component -")
print(f"but the distance transform has two separate peaks, and cutting at half")
print(f"the maximum recovers {n_seed-1} seeds, one per disk.")
print("That is what lets the watershed put a line between them.")
print()
print("Note the middle plot: the dip between the two humps is the neck where")
print("the disks meet. Whether the cut separates them depends on how deep that")
print("dip goes - which is exactly the judgement call in section 5.3.")


### 5.2 · Reading the next cell

Four steps, in order:

1. **Otsu, then opening.** Threshold gives a rough mask, and the opening deletes
   the speckle. On this clip that takes the mask from **79 connected components
   down to 34** — 45 specks removed — while giving up only 1.8 percentage points
   of area. Every speck you leave in becomes a spurious seed later.
2. **`sure_fg`** — the distance transform, cut at `0.3 * dist.max()`. These are
   the pixels deep enough inside a region to be confidently interior. They
   become the seeds.
3. **`sure_bg`** — the mask dilated outwards. **The name is misleading**: this is
   not the background, it is *everything that might still be foreground*. What
   lies **outside** it is the confident background. (This naming comes from the
   OpenCV tutorial; it trips nearly everyone the first time.)
4. **`unknown = sure_bg - sure_fg`** — the contested band between the two, which
   is exactly where the boundary must lie. These pixels get marker `0`, and the
   watershed decides who gets them.

So every pixel ends up in one of three states before flooding starts:
**seeded (2, 3, 4, …)**, **known background (1)**, or **contested (0)**.

In [ ]:
# ── Marker-controlled watershed ─────────────────────────────────────────────
t, bw = cv2.threshold(sm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, np.ones((5,5), np.uint8), iterations=2)

sure_bg = cv2.dilate(bw, np.ones((5,5), np.uint8), iterations=3)

#perform the distance transform
dist    = cv2.distanceTransform(bw, cv2.DIST_L2, 5)
#just keep the furthest pixels
_, sure_fg = cv2.threshold(dist, 0.3*dist.max(), 255, 0)   # <- the 0.3 is a CHOICE
sure_fg = sure_fg.astype(np.uint8)

#the pixels we didn't keep (removed by above threshold)
#are labelled grey (unknown)
unknown = cv2.subtract(sure_bg, sure_fg)

#perform connected components so we just have one label per region
n_markers, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0

ws = cv2.watershed(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR), markers.copy())
result = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB); result[ws == -1] = [255, 40, 40]

# The three-way marker map is the step you cannot otherwise see:
#   0 = contested (watershed decides)   1 = known background   2.. = seeds
marker_vis = np.zeros((*markers.shape, 3), np.uint8)
marker_vis[markers == 0] = [120, 120, 120]      # grey  - unknown
marker_vis[markers == 1] = [25,  35,  55]       # navy  - background
rng_m = np.random.default_rng(0)
for lbl in range(2, markers.max() + 1):
    marker_vis[markers == lbl] = rng_m.integers(90, 255, 3)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
panels = [(img, "1935 clip", 'gray'),
          (grad, "gradient magnitude\n(what flooding responds to)", 'magma'),
          (dist, "distance transform", 'viridis'),
          (marker_vis, f"markers: {n_markers-1} seeds\n+ background + unknown", None),
          (result, "marker-controlled result", None)]
for ax, (im, ttl, cm) in zip(axes, panels):
    ax.imshow(im, cmap=cm); ax.set_title(ttl, fontsize=10.5); ax.axis('off')
plt.tight_layout(); plt.show()

print("Note on panel 2: `grad` is shown for intuition - ridges are where the")
print("flood should stop - but it is NOT what we hand to cv2.watershed.")
print("cv2.watershed takes an IMAGE and computes its own gradient internally,")
print("so we pass `img`. Passing `grad` instead keeps the same 16 regions but")
print("moves the boundaries (only ~23% of boundary pixels coincide).")

regions = len(np.unique(ws)) - 1
print(f"markers seeded : {n_markers}")
print(f"regions found  : {regions}")
print(f"reduction      : {n_minima/max(regions,1):,.0f}x fewer than unguided")

### 5.3 · Where does `0.3` come from, and when does it break?

Nowhere — it is a knob, and it is the most consequential line in the cell. It
says *"a pixel is confidently interior if it is at least 30% as deep as the
deepest point anywhere in the image."* Run this to see what it costs you:

| cut | depth | seeds found |
|---|---|---|
| `0.1 × max` | 7.0 px | 34 |
| `0.2 × max` | 13.9 px | 17 |
| **`0.3 × max`** | **20.9 px** | **16** |
| `0.5 × max` | 34.8 px | 9 |
| `0.7 × max` | 48.7 px | 5 |

Compare the top row against step 1: the opened mask has **34 components**, and a
0.1 cut gives **34 seeds** — one each. By the 0.3 cut we are down to **16**. So
**18 real regions lost their seed** and will be absorbed into whichever
neighbour's flood reaches them first. They do not show up as errors; they simply
vanish into the region next door.

That is the failure mode to remember: **`dist.max()` is set by the single
deepest region in the image**, so one large blob raises the bar for everyone.
Small-but-genuine regions never clear it. If your regions vary a lot in size, a
single global fraction cannot serve all of them, and you want either a local
criterion (per-component peaks) or a fixed depth in pixels chosen from what you
know about the scene.

> **Worth trying:** change `0.3` to `0.15` in the cell above and re-run. You get
> many more regions — some of them real detail you were throwing away, some of
> them noise. There is no setting that is simply "correct", which is the honest
> lesson of this whole lecture.

---
## 6 · Evaluating a segmentation

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} \qquad\qquad
\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}$$

Never report pixel accuracy on a rare class — the next cell shows why.

In [ ]:
def iou(a, b):
    a, b = a.astype(bool), b.astype(bool)
    inter = (a & b).sum(); union = (a | b).sum()
    return inter/union if union else 0.0

def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    return 2*(a & b).sum() / (a.sum() + b.sum()) if (a.sum()+b.sum()) else 0.0

# the accuracy trap: an all-background "model" on a rare class
H = W = 800
for frac in (0.20, 0.05, 0.01):
    truth = np.zeros((H, W), bool)
    n = int(H*W*frac)
    truth.ravel()[:n] = True              # frac of pixels are the target
    pred = np.zeros((H, W), bool)         # predict nothing at all

    acc = (pred == truth).mean()
    print(f"target is {100*frac:4.1f}% of pixels  ->  "
          f"accuracy {100*acc:5.1f}%   IoU {iou(pred,truth):.3f}   Dice {dice(pred,truth):.3f}")

print()
print("Every one of those outputs an empty mask.")
print("Accuracy rewards ignoring the thing you are trying to find.")

---
## ✏️ Try at Home

**Estimated time: 15–20 minutes**

### Part A — Does Otsu improve after radiometric normalisation?

In AL2 you matched the 1935 and 1968 histograms onto 1958. Do that again here,
then re-run Otsu on the matched images.

1. Implement `match_histogram(src, ref)` (or paste your AL2 solution).
2. Match 1968 onto 1958.
3. Plot the histogram of the matched 1968 image. Is it bimodal now?
4. Does Otsu produce a more sensible threshold than the 143 it gave on the raw clip?

**Question:** does histogram matching create a valley that wasn't there, or does it
just move the same unimodal shape around? What does that tell you about what
radiometric normalisation can and cannot fix?

### Part B — Pick a threshold you can defend

Choose ONE epoch and segment the farm fields from everything else.

1. Try global Otsu, adaptive thresholding, and marker-controlled watershed.
2. For each, report the fraction of the image labelled "field".
3. Convert that to an area in hectares (the grid is 1 m/px, so 1 px = 1 m²,
   and 1 ha = 10,000 m²).
4. Write two sentences justifying which result you would put in a report — and
   say what you would need in order to know you were right.

> There is no ground truth here. Part of the answer is
> stating what evidence would settle it.


---
## Recommended Reading

| Source | Section | Notes |
|--------|---------|-------|
| **Szeliski, *Computer Vision: Algorithms and Applications* (2022)** | §7.5 | Segmentation. §3.1 covers thresholding as a point operator.|
| Gonzalez & Woods, *Digital Image Processing* (4th ed.) | Ch 10 | Full Otsu derivation, region growing, watershed |
| Otsu, N. (1979) | IEEE Trans. SMC 9(1), 62–66 | The original four-page paper — very readable |

**Videos**

| Video | Duration |
|-------|----------|
| Computerphile — *Thresholding and Otsu's Method* | 11 min |
| First Principles of Computer Vision — *Image Segmentation* | 18 min |

---

**Next:** Friday's **AL3** applies Canny, Otsu and watershed to your project imagery —
and project assignments are announced the same day.
